# TensorForge — Transformer model (Colab)

Fine-tunes a **multilingual transformer** (XLM-RoBERTa) on the RideEat tickets.
One shared model with **3 heads** predicts `category`, `secondary_category` and `is_urgent`
in a single pass.

At the end the model is exported to **ONNX** and compressed to **int8**, so it runs fast on the
judges' CPU-only server (2 vCPU, 4 GB RAM, no GPU, no internet).

### How to run
1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**
3. When asked, upload `train.jsonl` and `validation.jsonl` from the repo's `data/` folder
4. At the end, download `tensorforge_model.zip` (also saved to your Google Drive if you allow it)

Baseline to beat (TF-IDF + logistic regression, validation): **category accuracy 0.604, macro-F1 0.631**.

In [ ]:
%pip install -q "transformers>=4.44" onnx onnxruntime onnxscript scikit-learn

## 1. Settings

In [ ]:
import os
MODEL_NAME = "xlm-roberta-base"   # alternative to try: "intfloat/multilingual-e5-small" (smaller, faster)
MAX_LEN = 256          # longest train/validation ticket is 230 tokens
EPOCHS = 8
BATCH_SIZE = 16
LR = 3e-5
WARMUP = 0.1
SEED = 42

# After choosing the best epoch on validation, retrain on train + validation for the final model.
# Gives the model 20% more data. The validation scores printed below stay from the first run.
FINAL_TRAIN_ON_ALL = False

# Only for a quick local smoke test; leave False in Colab.
QUICK_TEST = os.environ.get("TF_QUICK_TEST") == "1"
if QUICK_TEST:
    MODEL_NAME = os.environ.get("TF_MODEL", MODEL_NAME)
    EPOCHS, MAX_LEN = 2, 64

## 2. Load the data

In [ ]:
import json, random
from pathlib import Path

try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DATA = Path("data")
DATA.mkdir(exist_ok=True)
missing = [f for f in ("train.jsonl", "validation.jsonl") if not (DATA / f).exists()]
if missing and IN_COLAB:
    from google.colab import files
    print("Upload:", missing)
    for name, content in files.upload().items():
        (DATA / Path(name).name).write_bytes(content)

def load(name):
    with open(DATA / name, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train, val = load("train.jsonl"), load("validation.jsonl")
if QUICK_TEST:
    random.Random(SEED).shuffle(train); random.Random(SEED).shuffle(val)
    train, val = train[:200], val[:100]
print(len(train), "train /", len(val), "validation tickets")

## 3. Inputs and labels

The model only sees `channel`, `subject` and `text`, because that is all the API receives.
`transformer_text` must stay identical to the copy in `app/features.py`.

In [ ]:
import numpy as np

CATEGORIES = ["payment_refund", "ride_trip_issue", "lost_item", "order_missing_wrong", "delivery_delay",
              "food_quality", "account_promo", "safety_conduct", "app_technical", "general_inquiry",
              "spam_irrelevant"]
SECONDARY = ["none"] + CATEGORIES   # "none" = no secondary category

def transformer_text(channel, subject, text):
    subject = (subject or "").strip()
    head = f"{channel} | {subject}" if subject else channel
    return f"{head}\n{text.strip()}"

def encode_rows(rows):
    X = [transformer_text(r["channel"], r.get("subject"), r["text"]) for r in rows]
    y_cat = np.array([CATEGORIES.index(r["category"]) for r in rows])
    y_sec = np.array([SECONDARY.index(r["secondary_category"] or "none") for r in rows])
    y_urg = np.array([float(r["is_urgent"]) for r in rows], dtype=np.float32)
    return X, y_cat, y_sec, y_urg

Xtr, ytr_cat, ytr_sec, ytr_urg = encode_rows(train)
Xva, yva_cat, yva_sec, yva_urg = encode_rows(val)
print(Xtr[0])

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

lengths = np.array([len(tokenizer(x)["input_ids"]) for x in Xtr + Xva])
print(f"tokens per ticket: median {np.median(lengths):.0f}, p95 {np.percentile(lengths, 95):.0f}, max {lengths.max()}")
print(f"tickets cut off at MAX_LEN={MAX_LEN}: {(lengths > MAX_LEN).mean():.1%}")

## 4. Model: one encoder, three heads

- **category**: 11 classes
- **secondary**: 12 classes (`none` + the 11 categories)
- **urgent**: one yes/no output

The encoder output is mean-pooled over the tokens, then each head is a small linear layer.
Class weights make rare classes (safety, spam, urgent) count more during training.

In [ ]:
import torch, torch.nn as nn
from transformers import AutoModel

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

class TicketModel(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.cat_head = nn.Linear(h, len(CATEGORIES))
        self.sec_head = nn.Linear(h, len(SECONDARY))
        self.urg_head = nn.Linear(h, 1)

    def forward(self, input_ids, attention_mask):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        pooled = self.dropout((hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-6))
        return self.cat_head(pooled), self.sec_head(pooled), self.urg_head(pooled).squeeze(-1)

def class_weights(y, n):
    counts = np.bincount(y, minlength=n).astype(np.float32) + 1
    w = (counts.sum() / counts) ** 0.5     # softened inverse frequency
    return torch.tensor(w / w.mean(), dtype=torch.float32)

In [ ]:
from torch.utils.data import DataLoader, Dataset

class Tickets(Dataset):
    def __init__(self, X, y_cat, y_sec, y_urg):
        self.X, self.y_cat, self.y_sec, self.y_urg = X, y_cat, y_sec, y_urg
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y_cat[i], self.y_sec[i], self.y_urg[i]

def collate(batch):
    X, c, s, u = zip(*batch)
    enc = tokenizer(list(X), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    return enc, torch.tensor(c), torch.tensor(s), torch.tensor(u)

@torch.no_grad()
def predict_probs(model, X, batch_size=64):
    model.eval()
    cat, sec, urg = [], [], []
    for i in range(0, len(X), batch_size):
        enc = tokenizer(X[i:i + batch_size], truncation=True, max_length=MAX_LEN, padding=True,
                        return_tensors="pt").to(device)
        with torch.autocast(device_type=device, enabled=device == "cuda"):
            c, s, u = model(enc["input_ids"], enc["attention_mask"])
        cat.append(c.float().softmax(-1).cpu()); sec.append(s.float().softmax(-1).cpu())
        urg.append(u.float().sigmoid().cpu())
    return torch.cat(cat).numpy(), torch.cat(sec).numpy(), torch.cat(urg).numpy()

## 5. Turning probabilities into API answers

Same rules as the API (`app/model.py`):
- secondary is the most likely label **other than the primary**
- spam is never urgent and has no secondary

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def decide(cat_p, sec_p, urg_p, threshold):
    cat = cat_p.argmax(1)
    sec_p = sec_p.copy()
    sec_p[np.arange(len(cat)), cat + 1] = -1          # +1: SECONDARY[0] is "none"
    sec = sec_p.argmax(1)
    urg = urg_p >= threshold
    spam = cat == CATEGORIES.index("spam_irrelevant")
    sec[spam], urg[spam] = 0, False
    return cat, sec, urg

def best_threshold(y, p):
    grid = np.arange(0.05, 0.96, 0.025)
    return float(max(grid, key=lambda t: f1_score(y, p >= t, zero_division=0)))

def scores(probs, y_cat, y_sec, y_urg, threshold):
    cat, sec, urg = decide(*probs, threshold)
    return {
        "category_acc": accuracy_score(y_cat, cat),
        "category_f1": f1_score(y_cat, cat, average="macro"),
        "secondary_acc": accuracy_score(y_sec, sec),
        "secondary_f1": f1_score(y_sec, sec, average="macro"),
        "urgent_f1": f1_score(y_urg, urg, zero_division=0),
        "all_three": float(np.mean((cat == y_cat) & (sec == y_sec) & (urg == y_urg))),
    }

## 6. Train

In [ ]:
import copy, time
from transformers import get_linear_schedule_with_warmup

def train_model(X, y_cat, y_sec, y_urg, epochs, eval_data=None):
    model = TicketModel(MODEL_NAME).to(device)
    loader = DataLoader(Tickets(X, y_cat, y_sec, y_urg), batch_size=BATCH_SIZE, shuffle=True,
                        collate_fn=collate, generator=torch.Generator().manual_seed(SEED))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = len(loader) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler(enabled=device == "cuda")

    ce_cat = nn.CrossEntropyLoss(weight=class_weights(y_cat, len(CATEGORIES)).to(device))
    ce_sec = nn.CrossEntropyLoss(weight=class_weights(y_sec, len(SECONDARY)).to(device))
    pos = (len(y_urg) - y_urg.sum()) / max(y_urg.sum(), 1)
    bce_urg = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos ** 0.5, device=device))

    history, best = [], None
    for epoch in range(1, epochs + 1):
        model.train(); t0 = time.time(); total = 0.0
        for enc, c, s, u in loader:
            enc, c, s, u = enc.to(device), c.to(device), s.to(device), u.to(device)
            with torch.autocast(device_type=device, enabled=device == "cuda"):
                lc, ls, lu = model(enc["input_ids"], enc["attention_mask"])
                loss = ce_cat(lc.float(), c) + 0.5 * ce_sec(ls.float(), s) + 0.5 * bce_urg(lu.float(), u)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            total += loss.item()

        line = f"epoch {epoch}: loss {total / len(loader):.3f} ({time.time() - t0:.0f}s)"
        if eval_data is not None:
            probs = predict_probs(model, eval_data[0])
            th = best_threshold(eval_data[3], probs[2])
            sc = scores(probs, *eval_data[1:], th)
            sc.update(epoch=epoch, threshold=th)
            history.append(sc)
            line += (f" | cat acc {sc['category_acc']:.3f} F1 {sc['category_f1']:.3f}"
                     f" | sec F1 {sc['secondary_f1']:.3f} | urg F1 {sc['urgent_f1']:.3f}")
            # Pick the epoch by category macro-F1, with urgency and secondary as tie-breakers.
            key = sc["category_f1"] + 0.3 * sc["urgent_f1"] + 0.2 * sc["secondary_f1"]
            if best is None or key > best[0]:
                best = (key, sc, copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()}))
        print(line)

    if best is not None:
        model.load_state_dict(best[2])
        return model, best[1], history
    return model, None, history

model, best, history = train_model(Xtr, ytr_cat, ytr_sec, ytr_urg, EPOCHS,
                                   eval_data=(Xva, yva_cat, yva_sec, yva_urg))
print("\nbest epoch:", best["epoch"], {k: round(v, 3) for k, v in best.items()})

## 7. Validation report

In [ ]:
URGENT_THRESHOLD = best["threshold"]
val_probs = predict_probs(model, Xva)
cat, sec, urg = decide(*val_probs, URGENT_THRESHOLD)

print(f"category  acc {accuracy_score(yva_cat, cat):.3f}  macro-F1 {f1_score(yva_cat, cat, average='macro'):.3f}"
      f"   (baseline 0.604 / 0.631)")
print(f"secondary acc {accuracy_score(yva_sec, sec):.3f}  macro-F1 {f1_score(yva_sec, sec, average='macro'):.3f}")
print(f"urgent    F1 {f1_score(yva_urg, urg):.3f}  (threshold {URGENT_THRESHOLD:.3f})")
print()
print(classification_report(yva_cat, cat, labels=range(len(CATEGORIES)), target_names=CATEGORIES,
                            digits=3, zero_division=0))

langs = np.array([r["language"] for r in val])   # language is used only to report, never as input
print("category accuracy by language:")
for lang in ["en", "singlish", "si", "ta", "tanglish", "mixed"]:
    m = langs == lang
    if m.any():
        print(f"  {lang:9s} n={m.sum():4d}  acc {accuracy_score(yva_cat[m], cat[m]):.3f}")

## 8. (Optional) Final model on train + validation

Only runs when `FINAL_TRAIN_ON_ALL = True`. Uses the best epoch count and urgency threshold
found above. Its validation score can no longer be measured, so report the numbers from step 7.

In [ ]:
if FINAL_TRAIN_ON_ALL:
    Xall = Xtr + Xva
    model, _, _ = train_model(Xall, np.concatenate([ytr_cat, yva_cat]), np.concatenate([ytr_sec, yva_sec]),
                              np.concatenate([ytr_urg, yva_urg]), best["epoch"])
    print("final model trained on", len(Xall), "tickets")

## 9. Export to ONNX and compress to int8

The API server has no GPU, so the model is converted to ONNX (runs with `onnxruntime`, no PyTorch
needed) and its weights are compressed from 32-bit to 8-bit, making it about 4x smaller and faster.

In [ ]:
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

OUT = Path("tensorforge_model")
OUT.mkdir(exist_ok=True)

class Export(nn.Module):
    # Returns probabilities so the API does not need to re-implement softmax/sigmoid.
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask):
        c, s, u = self.m(input_ids, attention_mask)
        return c.softmax(-1), s.softmax(-1), u.sigmoid()

export = Export(model).cpu().float().eval()
dummy = tokenizer(["chat\nhello"], return_tensors="pt")
torch.onnx.export(
    export, (dummy["input_ids"], dummy["attention_mask"]), str(OUT / "model_fp32.onnx"),
    input_names=["input_ids", "attention_mask"], output_names=["category", "secondary", "urgent"],
    dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"},
                  "category": {0: "batch"}, "secondary": {0: "batch"}, "urgent": {0: "batch"}},
    opset_version=17, dynamo=False,
)
quantize_dynamic(str(OUT / "model_fp32.onnx"), str(OUT / "model.onnx"), weight_type=QuantType.QInt8)
(OUT / "model_fp32.onnx").unlink()
print(f"model.onnx: {(OUT / 'model.onnx').stat().st_size / 1e6:.0f} MB")

## 10. Check the int8 model on CPU (same as the judges' server)

In [ ]:
opts = ort.SessionOptions()
opts.intra_op_num_threads = 2          # the judges' reference server has 2 vCPU
session = ort.InferenceSession(str(OUT / "model.onnx"), opts, providers=["CPUExecutionProvider"])

def onnx_probs(X, batch_size=32):
    outs = [[], [], []]
    for i in range(0, len(X), batch_size):
        enc = tokenizer(X[i:i + batch_size], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="np")
        res = session.run(None, {"input_ids": enc["input_ids"].astype(np.int64),
                                 "attention_mask": enc["attention_mask"].astype(np.int64)})
        for o, r in zip(outs, res):
            o.append(r)
    return tuple(np.concatenate(o) for o in outs)

t0 = time.time()
q_probs = onnx_probs(Xva)
per_ticket = (time.time() - t0) / len(Xva)
q = scores(q_probs, yva_cat, yva_sec, yva_urg, URGENT_THRESHOLD)
print("int8 ONNX on validation:", {k: round(v, 3) for k, v in q.items()})
if not FINAL_TRAIN_ON_ALL:
    print("PyTorch on validation:  ", {k: round(v, 3) for k, v in scores(val_probs, yva_cat, yva_sec, yva_urg, URGENT_THRESHOLD).items()})
print(f"CPU speed: {per_ticket * 1000:.0f} ms per ticket in batches"
      f" -> 5,000 tickets in about {per_ticket * 5000 / 60:.1f} min (limit 30 min)")

t0 = time.time()
for x in Xva[:20]:
    onnx_probs([x])
print(f"single ticket: {(time.time() - t0) / 20 * 1000:.0f} ms (limit 1000 ms)")

## 11. Save the package

In [ ]:
import hashlib, shutil

tokenizer.save_pretrained(OUT)
config = {
    "name": "xlmr" if "xlm-roberta" in MODEL_NAME else MODEL_NAME.split("/")[-1],
    "base_model": MODEL_NAME,
    "text_format": "transformer_text_v1",
    "max_len": MAX_LEN,
    "categories": CATEGORIES,
    "secondary": SECONDARY,
    "urgent_threshold": URGENT_THRESHOLD,
    "trained_on": "train+validation" if FINAL_TRAIN_ON_ALL else "train",
    "best_epoch": best["epoch"],
    "validation_scores": {k: round(float(v), 4) for k, v in best.items()},
    "int8_validation_scores": {k: round(float(v), 4) for k, v in q.items()},
    "model_sha256": hashlib.sha256((OUT / "model.onnx").read_bytes()).hexdigest(),
}
(OUT / "config.json").write_text(json.dumps(config, indent=2))
(OUT / "history.json").write_text(json.dumps(history, indent=2, default=float))

shutil.make_archive("tensorforge_model", "zip", OUT)
print(json.dumps(config, indent=2))
print(f"\ntensorforge_model.zip: {Path('tensorforge_model.zip').stat().st_size / 1e6:.0f} MB")

In [ ]:
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        dest = Path("/content/drive/MyDrive/tensorforge")
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copy("tensorforge_model.zip", dest / "tensorforge_model.zip")
        print("Saved to Google Drive:", dest / "tensorforge_model.zip")
    except Exception as e:
        print("Drive copy skipped:", e)
    from google.colab import files
    files.download("tensorforge_model.zip")